# 03 – Guiding Questions
Self-contained: reads only the three processed files, rebuilds everything it needs, and runs top to bottom.

Needs in `../data/processed/`: `transactions_clean.csv`, `flight_costs_df.csv`, `gates_clean.csv`.

Each question has: code -> output -> a **Your answer** cell for you to write the takeaway.

In [1]:
import pandas as pd, numpy as np, altair as alt
alt.data_transformers.disable_max_rows()
pd.options.display.float_format = "{:,.2f}".format
pd.options.display.width = 220
pd.options.display.max_columns = 30

tx    = pd.read_csv("../data/processed/transactions_clean.csv", parse_dates=["Transaction_Date"])
fc    = pd.read_csv("../data/processed/flight_costs_df.csv",    parse_dates=["Cost_Date"])
gates = pd.read_csv("../data/processed/gates_clean.csv")

def fiscal_year(d):                       # FY runs Sep 1 - Aug 31, labelled by the year it ends
    return d.dt.year + (d.dt.month >= 9).astype(int)

tx["Fiscal_Year"] = fiscal_year(tx["Transaction_Date"])
tx["Month"]       = tx["Transaction_Date"].dt.to_period("M").astype(str)
fc["Fiscal_Year"] = fiscal_year(fc["Cost_Date"])
fc["Month"]       = fc["Cost_Date"].dt.to_period("M").astype(str)
fc["Origin"], fc["Dest"] = fc["Route_ID"].str[:3], fc["Route_ID"].str[4:]

HUBS = ["YVR", "YYC", "YYZ", "YUL"]

# Connecting vs direct booking (a booking with >1 distinct route is connecting)
n_routes = tx.groupby("Booking_ID")["Route_ID"].nunique()
tx["Booking_Type"] = np.where(tx["Booking_ID"].map(n_routes) > 1, "Connecting", "Direct")

assert tx["Transaction_Date"].between("2024-09-01", "2026-08-31 23:59:59").all()
print(tx.shape, fc.shape, gates.shape)

(558129, 12) (28801, 13) (112, 5)


### Gate cost set-up (lease years run Jan 1 - Jan 1, fiscal years run Sep 1 - Aug 31)

In [2]:
# One row per physical gate (a gate shared by several routes appears several times in gates.csv)
gate_unique = gates.drop_duplicates(["Year", "City", "Gate_Number"])

gate_annual = (gate_unique.pivot_table(index="City", columns="Year",
                                        values="Annual_Gate_Cost", aggfunc="sum")
               .rename(columns={"2024-2025": "Lease_2425", "2025-2026": "Lease_2526"}))
gate_annual["Growth"]         = gate_annual["Lease_2526"] / gate_annual["Lease_2425"] - 1
gate_annual["Lease_2627_est"] = gate_annual["Lease_2526"] * (1 + gate_annual["Growth"])   # assumption: each airport repeats its own growth

# Fiscal-year gate cost: FY25 = Sep-Dec 2024 (lease 24-25) + Jan-Aug 2025 (lease 25-26)
#                        FY26 = Sep-Dec 2025 (lease 25-26) + Jan-Aug 2026 (lease 26-27, estimated)
gate_fy = pd.DataFrame({
    2025: gate_annual["Lease_2425"] * 4/12 + gate_annual["Lease_2526"]     * 8/12,
    2026: gate_annual["Lease_2526"] * 4/12 + gate_annual["Lease_2627_est"] * 8/12,
})

# Split each airport's gate cost evenly across flights departing from it
flights_by_airport = fc.groupby(["Fiscal_Year", "From"]).size().rename("Flights").reset_index()
gate_long = gate_fy.stack().rename("Gate_Cost").reset_index()
gate_long.columns = ["City", "Fiscal_Year", "Gate_Cost"]
alloc = gate_long.merge(flights_by_airport, left_on=["City", "Fiscal_Year"],
                        right_on=["From", "Fiscal_Year"], validate="one_to_one")
alloc["Gate_Per_Flight"] = alloc["Gate_Cost"] / alloc["Flights"]

fc = fc.merge(alloc[["From", "Fiscal_Year", "Gate_Per_Flight"]],
              on=["From", "Fiscal_Year"], how="left", validate="many_to_one")
fc["Total_With_Gates"] = fc["Total"] + fc["Gate_Per_Flight"]
assert fc["Gate_Per_Flight"].notna().all()
print("FY gate totals:", gate_fy.sum().round(0).to_dict())

FY gate totals: {2025: 9806600.0, 2026: 10023126.0}


---
## Q1. How did revenue and costs change between FY2024-25 and FY2025-26? When did results start to decline, and which changes explain most of the drop?

**1a. Fiscal-year summary**

In [3]:
fy_summary = pd.DataFrame({
    "Revenue":      tx.groupby("Fiscal_Year")["Amount"].sum(),
    "Flight_Costs": fc.groupby("Fiscal_Year")["Total"].sum(),
    "Gate_Costs":   gate_fy.sum(),
    "Flights":      fc.groupby("Fiscal_Year").size(),
})
fy_summary["Profit_Before_Gates"] = fy_summary["Revenue"] - fy_summary["Flight_Costs"]
fy_summary["Profit_After_Gates"]  = fy_summary["Profit_Before_Gates"] - fy_summary["Gate_Costs"]
fy_summary["Margin_After_Gates_%"] = fy_summary["Profit_After_Gates"] / fy_summary["Revenue"] * 100
fy_summary["Revenue_per_Flight"]  = fy_summary["Revenue"] / fy_summary["Flights"]
fy_summary["Cost_per_Flight"]     = fy_summary["Flight_Costs"] / fy_summary["Flights"]

q1a = fy_summary.T.rename(columns={2025: "FY2024-25", 2026: "FY2025-26"})
q1a["Change"]   = q1a["FY2025-26"] - q1a["FY2024-25"]
q1a["Change_%"] = q1a["Change"] / q1a["FY2024-25"].abs() * 100
q1a

,FY2024-25,FY2025-26,Change,Change_%
Revenue,"180,629,746.30","189,999,278.55","9,369,532.25",5.19
Flight_Costs,"192,985,563.99","238,376,476.25","45,390,912.26",23.52
Gate_Costs,"9,806,600.00","10,023,125.91","216,525.91",2.21
Flights,"13,072.00","15,729.00","2,657.00",20.33
Profit_Before_Gates,"-12,355,817.69","-48,377,197.70","-36,021,380.01",-291.53
Profit_After_Gates,"-22,162,417.69","-58,400,323.61","-36,237,905.92",-163.51
Margin_After_Gates_%,-12.27,-30.74,-18.47,-150.52
Revenue_per_Flight,"13,818.07","12,079.55","-1,738.51",-12.58
Cost_per_Flight,"14,763.28","15,155.22",391.94,2.65


**1b. Month by month: when did it turn?** (gate cost per month = that calendar year's lease / 12)

In [4]:
monthly = pd.DataFrame({
    "Revenue":      tx.groupby("Month")["Amount"].sum(),
    "Flight_Costs": fc.groupby("Month")["Total"].sum(),
    "Flights":      fc.groupby("Month").size(),
})
lease_by_year = {2024: gate_annual["Lease_2425"].sum(),
                 2025: gate_annual["Lease_2526"].sum(),
                 2026: gate_annual["Lease_2627_est"].sum()}
monthly["Gate_Costs"]          = [lease_by_year[int(m[:4])] / 12 for m in monthly.index]
monthly["Profit_Before_Gates"] = monthly["Revenue"] - monthly["Flight_Costs"]
monthly["Profit_After_Gates"]  = monthly["Profit_Before_Gates"] - monthly["Gate_Costs"]
monthly["Margin_Before_%"]     = monthly["Profit_Before_Gates"] / monthly["Revenue"] * 100
monthly["Margin_After_%"]      = monthly["Profit_After_Gates"]  / monthly["Revenue"] * 100

print("First month with a loss BEFORE gates:", monthly.index[monthly["Profit_Before_Gates"] < 0][0])
print("First month with a loss AFTER gates: ", monthly.index[monthly["Profit_After_Gates"]  < 0][0])
monthly

First month with a loss BEFORE gates: 2025-03
First month with a loss AFTER gates:  2025-02


,Revenue,Flight_Costs,Flights,Gate_Costs,Profit_Before_Gates,Profit_After_Gates,Margin_Before_%,Margin_After_%
Month,,,,,,,,
2024-09,"13,907,307.03","11,716,037.65",833,"805,366.67","2,191,269.38","1,385,902.71",15.76,9.97
2024-10,"14,011,803.98","12,284,511.15",873,"805,366.67","1,727,292.83","921,926.16",12.33,6.58
2024-11,"13,740,580.23","12,547,354.26",793,"805,366.67","1,193,225.97","387,859.30",8.68,2.82
2024-12,"15,763,715.10","13,810,985.02",928,"805,366.67","1,952,730.08","1,147,363.41",12.39,7.28
2025-01,"14,356,457.27","13,362,980.18",802,"823,141.67","993,477.09","170,335.42",6.92,1.19
2025-02,"13,658,478.38","13,043,794.51",792,"823,141.67","614,683.87","-208,457.80",4.50,-1.53
2025-03,"14,730,133.16","15,759,466.26",1157,"823,141.67","-1,029,333.10","-1,852,474.77",-6.99,-12.58
2025-04,"13,596,549.87","16,910,203.58",1246,"823,141.67","-3,313,653.71","-4,136,795.38",-24.37,-30.43
2025-05,"14,964,947.96","18,728,234.60",1308,"823,141.67","-3,763,286.64","-4,586,428.31",-25.15,-30.65


In [5]:
m = monthly.reset_index().rename(columns={"index": "Month"})
m_long = m.melt(id_vars="Month", value_vars=["Margin_Before_%", "Margin_After_%"],
                var_name="Measure", value_name="Margin_%")
line = alt.Chart(m_long).mark_line(point=True).encode(
    x=alt.X("Month:O", axis=alt.Axis(labelAngle=-60)), y="Margin_%:Q", color="Measure:N",
    tooltip=["Month", "Measure", alt.Tooltip("Margin_%:Q", format=".1f")])
zero = alt.Chart(pd.DataFrame({"y": [0]})).mark_rule(strokeDash=[4, 4], color="gray").encode(y="y:Q")
(line + zero).properties(title="Monthly operating margin", width=700, height=300)

alt.LayerChart(...)

**1c. Is it more flights, or more expensive flights?** (volume vs per-flight cost effect, exact decomposition)

In [6]:
cats = ["Fuel", "Crew", "Maintenance", "Food"]
n_fl = fc.groupby("Fiscal_Year").size()
per_flight = fc.groupby("Fiscal_Year")[cats].mean()

volume_effect = (n_fl[2026] - n_fl[2025]) * per_flight.loc[2025]       # extra flights at last year's cost per flight
rate_effect   = n_fl[2026] * (per_flight.loc[2026] - per_flight.loc[2025])   # higher/lower cost per flight on this year's flights

decomp = pd.DataFrame({
    "FY25_Cost":   fc[fc.Fiscal_Year == 2025][cats].sum(),
    "FY26_Cost":   fc[fc.Fiscal_Year == 2026][cats].sum(),
    "Volume_Effect (more flights)":   volume_effect,
    "Rate_Effect (cost per flight)":  rate_effect,
    "Cost_per_Flight_Change_%": (per_flight.loc[2026] / per_flight.loc[2025] - 1) * 100,
})
decomp["Total_Change"] = decomp["FY26_Cost"] - decomp["FY25_Cost"]
decomp.loc["TOTAL"] = decomp.sum()
decomp.loc["TOTAL", "Cost_per_Flight_Change_%"] = (fc[fc.Fiscal_Year == 2026]["Total"].mean() /
                                                    fc[fc.Fiscal_Year == 2025]["Total"].mean() - 1) * 100
print(f"Flights: {n_fl[2025]:,} -> {n_fl[2026]:,}  ({n_fl[2026]/n_fl[2025]-1:+.1%})")
print(f"Share of flight-cost increase explained by MORE FLIGHTS: "
      f"{decomp.loc['TOTAL','Volume_Effect (more flights)'] / decomp.loc['TOTAL','Total_Change']:.0%}")
decomp

Flights: 13,072 -> 15,729  (+20.3%)
Share of flight-cost increase explained by MORE FLIGHTS: 86%


,FY25_Cost,FY26_Cost,Volume_Effect (more flights),Rate_Effect (cost per flight),Cost_per_Flight_Change_%,Total_Change
Fuel,"67,061,083.08","83,182,330.22","13,630,760.23","2,490,486.91",3.09,"16,121,247.14"
Crew,"59,717,729.49","74,767,150.35","12,138,158.45","2,911,262.41",4.05,"15,049,420.86"
Maintenance,"58,384,123.71","71,387,479.95","11,867,091.24","1,136,265.00",1.62,"13,003,356.24"
Food,"7,822,627.71","9,039,515.73","1,590,018.50","-373,130.48",-3.96,"1,216,888.02"
TOTAL,"192,985,563.99","238,376,476.25","39,226,028.42","6,164,883.84",2.65,"45,390,912.26"


**1d. Profit bridge FY25 -> FY26** (every line is its impact on profit after gates; they sum exactly to the change)

In [7]:
rev_comp = tx.pivot_table(index="Fiscal_Year", columns="Transaction_Type", values="Amount", aggfunc="sum")

bridge = {f"{c} revenue": rev_comp.loc[2026, c] - rev_comp.loc[2025, c] for c in rev_comp.columns}
bridge["More flights (volume of costs)"] = -volume_effect.sum()
for c in cats:
    bridge[f"Higher {c.lower()} cost per flight"] = -rate_effect[c]
bridge["Gate leases"] = -(gate_fy[2026].sum() - gate_fy[2025].sum())

bridge = pd.Series(bridge, name="Profit_Impact")
delta_profit = fy_summary.loc[2026, "Profit_After_Gates"] - fy_summary.loc[2025, "Profit_After_Gates"]
assert abs(bridge.sum() - delta_profit) < 1, "bridge does not reconcile"

q1d = bridge.sort_values().to_frame()
q1d["Share_of_Decline_%"] = q1d["Profit_Impact"] / delta_profit * 100
print(f"Total change in profit after gates: {delta_profit:,.0f}")
q1d

Total change in profit after gates: -36,237,906


,Profit_Impact,Share_of_Decline_%
More flights (volume of costs),"-39,226,028.42",108.25
Higher crew cost per flight,"-2,911,262.41",8.03
Higher fuel cost per flight,"-2,490,486.91",6.87
Refund revenue,"-1,149,131.33",3.17
Higher maintenance cost per flight,"-1,136,265.00",3.14
Gate leases,"-216,525.91",0.60
Change Fee revenue,"45,850.00",-0.13
Seat Upgrade revenue,"50,975.01",-0.14
Food & Onboard revenue,"58,114.20",-0.16
Extra Baggage revenue,"137,599.91",-0.38


In [36]:
alt.Chart(q1d.reset_index().rename(columns={"index": "Driver"})).mark_bar().encode(
    y=alt.Y("Driver:N", sort="x"),
    x=alt.X("Profit_Impact:Q", axis=alt.Axis(format="~s"), title="Impact on profit ($)"),
    color=alt.condition(alt.datum.Profit_Impact >= 0, alt.value("#2A9D8F"), alt.value("#E76F51")),
    tooltip=["Driver", alt.Tooltip("Profit_Impact:Q", format=",.0f")]
).properties(title="What explains the drop in profit, FY25 -> FY26", width=600, height=320)

alt.Chart(...)

**Your answer (Q1):**

- Revenue/cost change: 
- When results started to decline: 
- Which changes explain most of the drop: 

---
## Q2. How does financial performance differ across the network? Are some parts of the business carrying others?

In [9]:
route_rev  = tx.groupby(["Fiscal_Year", "Route_ID"])["Amount"].sum().rename("Revenue")
route_cost = fc.groupby(["Fiscal_Year", "Route_ID"]).agg(
    Flights=("Total", "size"), Flight_Costs=("Total", "sum"), Gate_Costs=("Gate_Per_Flight", "sum"))

routes = pd.concat([route_rev, route_cost], axis=1).fillna(0).reset_index()
routes["Profit_Before_Gates"] = routes["Revenue"] - routes["Flight_Costs"]
routes["Profit_After_Gates"]  = routes["Profit_Before_Gates"] - routes["Gate_Costs"]
routes["Margin_%"]            = routes["Profit_After_Gates"] / routes["Revenue"] * 100
routes["Profit_per_Flight"]   = routes["Profit_After_Gates"] / routes["Flights"]
routes["Origin"], routes["Dest"] = routes["Route_ID"].str[:3], routes["Route_ID"].str[4:]
routes["Route_Type"] = np.select(
    [routes.Origin.isin(HUBS) & routes.Dest.isin(HUBS), routes.Origin.isin(HUBS) | routes.Dest.isin(HUBS)],
    ["Hub-Hub", "Hub-Regional"], default="Regional-Regional")

r26 = routes[routes.Fiscal_Year == 2026].copy()
r25 = routes[routes.Fiscal_Year == 2025].copy()
print("Routes:", r26.Route_ID.nunique())

Routes: 54


**2a. Hub vs regional**

In [10]:
by_type = routes.groupby(["Fiscal_Year", "Route_Type"]).agg(
    Routes=("Route_ID", "nunique"), Flights=("Flights", "sum"), Revenue=("Revenue", "sum"),
    Flight_Costs=("Flight_Costs", "sum"), Gate_Costs=("Gate_Costs", "sum"),
    Profit_After_Gates=("Profit_After_Gates", "sum"))
by_type["Margin_%"] = by_type["Profit_After_Gates"] / by_type["Revenue"] * 100
by_type

Routes  Flights       Revenue   Flight_Costs   Gate_Costs  Profit_After_Gates  Margin_%
Fiscal_Year Route_Type                                                                                                
2025        Hub-Hub                10     3038 90,507,618.72  94,055,822.80 2,343,591.25       -5,891,795.33     -6.51
            Hub-Regional           38     9415 83,996,975.55  91,174,441.59 6,761,175.07      -13,938,641.11    -16.59
            Regional-Regional       6      619  6,125,152.03   7,755,299.60   701,833.68       -2,331,981.25    -38.07
2026        Hub-Hub                10     3518 93,951,342.02 115,134,518.49 2,342,946.90      -23,526,123.37    -25.04
            Hub-Regional           38    11213 89,061,251.09 112,962,282.58 6,913,757.22      -30,814,788.71    -34.60
            Regional-Regional       6      998  6,986,685.44  10,279,675.18   766,421.79       -4,059,411.53    -58.10

**2b. Who is carrying whom?** (profitable routes vs loss-making routes, FY26)

In [11]:
for label, col in [("before gates", "Profit_Before_Gates"), ("after gates", "Profit_After_Gates")]:
    win, lose = r26[r26[col] > 0], r26[r26[col] < 0]
    print(f"[{label}] profitable routes: {len(win)} of {len(r26)} | profit earned: {win[col].sum():>14,.0f} "
          f"| losses: {lose[col].sum():>14,.0f}")

bottom10 = r26.nsmallest(10, "Profit_After_Gates")
print(f"\nBottom 10 routes = {bottom10.Profit_After_Gates.sum() / r26[r26.Profit_After_Gates<0].Profit_After_Gates.sum():.0%} of all route losses")
cols = ["Route_ID", "Route_Type", "Flights", "Revenue", "Flight_Costs", "Gate_Costs", "Profit_After_Gates", "Margin_%", "Profit_per_Flight"]
print("\nTOP 10 (carrying the network):");  display(r26.nlargest(10, "Profit_After_Gates")[cols])
print("BOTTOM 10:");                          display(bottom10[cols])

[before gates] profitable routes: 26 of 54 | profit earned:     12,557,385 | losses:    -60,934,583
[after gates] profitable routes: 24 of 54 | profit earned:      8,410,360 | losses:    -66,810,684

Bottom 10 routes = 77% of all route losses

TOP 10 (carrying the network):


,Route_ID,Route_Type,Flights,Revenue,Flight_Costs,Gate_Costs,Profit_After_Gates,Margin_%,Profit_per_Flight
105,YYZ-YYC,Hub-Hub,176,"9,989,722.20","8,483,619.49","104,293.02","1,401,809.69",14.03,"7,964.83"
95,YYC-YYZ,Hub-Hub,173,"8,714,069.85","7,516,284.40","115,631.03","1,082,154.42",12.42,"6,255.23"
91,YYC-YVR,Hub-Hub,350,"7,140,524.80","5,892,329.61","233,935.62","1,014,259.57",14.20,"2,897.88"
78,YVR-YYC,Hub-Hub,334,"6,403,331.12","5,170,143.98","270,821.26","962,365.88",15.03,"2,881.33"
102,YYZ-YVR,Hub-Hub,296,"18,374,194.46","17,442,040.03","175,401.90","756,752.53",4.12,"2,556.60"
80,YVR-YYZ,Hub-Hub,307,"16,899,461.09","16,069,621.22","248,928.53","580,911.34",3.44,"1,892.22"
73,YVR-YEG,Hub-Regional,208,"4,104,143.04","3,587,510.75","168,655.16","347,977.13",8.48,"1,672.97"
54,YEG-YVR,Hub-Regional,212,"4,405,588.85","3,980,193.10","157,348.61","268,047.14",6.08,"1,264.37"
103,YYZ-YWG,Hub-Regional,125,"4,027,439.48","3,690,385.14","74,071.75","262,982.59",6.53,"2,103.86"
92,YYC-YWG,Hub-Regional,116,"2,887,379.33","2,600,611.71","77,532.95","209,234.67",7.25,"1,803.75"


BOTTOM 10:


,Route_ID,Route_Type,Flights,Revenue,Flight_Costs,Gate_Costs,Profit_After_Gates,Margin_%,Profit_per_Flight
71,YUL-YVR,Hub-Hub,306,"9,317,689.66","19,629,342.96","184,661.24","-10,496,314.54",-112.65,"-34,301.68"
75,YVR-YUL,Hub-Hub,314,"8,495,080.07","17,889,173.63","254,604.42","-9,648,697.98",-113.58,"-30,728.34"
60,YHZ-YYZ,Hub-Regional,372,"4,768,353.23","9,764,294.01","269,601.90","-5,265,542.68",-110.43,"-14,154.68"
72,YUL-YYZ,Hub-Hub,628,"4,427,618.74","8,791,986.88","378,977.97","-4,743,346.11",-107.13,"-7,553.10"
101,YYZ-YUL,Hub-Hub,634,"4,189,650.03","8,249,976.29","375,691.90","-4,436,018.16",-105.88,"-6,996.87"
99,YYZ-YHZ,Hub-Regional,344,"3,977,148.99","8,129,783.93","203,845.45","-4,356,480.39",-109.54,"-12,664.19"
100,YYZ-YOW,Hub-Regional,1062,"3,305,145.75","6,798,697.29","629,313.57","-4,122,865.11",-124.74,"-3,882.17"
64,YOW-YYZ,Hub-Regional,1038,"3,456,661.62","7,157,163.61","291,580.92","-3,992,082.91",-115.49,"-3,845.94"
58,YHZ-YUL,Hub-Regional,224,"2,009,227.95","4,203,219.47","162,340.93","-2,356,332.45",-117.28,"-10,519.34"
69,YUL-YOW,Hub-Regional,546,"958,387.74","2,952,225.31","329,493.58","-2,323,331.15",-242.42,"-4,255.19"


In [12]:
alt.Chart(r26).mark_bar().encode(
    y=alt.Y("Route_ID:N", sort="x"), x=alt.X("Profit_After_Gates:Q", axis=alt.Axis(format="~s")),
    color=alt.condition(alt.datum.Profit_After_Gates >= 0, alt.value("#2A9D8F"), alt.value("#E76F51")),
    tooltip=["Route_ID", "Route_Type", "Flights", alt.Tooltip("Profit_After_Gates:Q", format=",.0f")]
).properties(title="FY2026 profit after gates, all routes", width=600, height=700)

alt.Chart(...)

**2c. By departure airport** (gate cost is charged at the departure airport)

In [13]:
by_origin = r26.groupby("Origin").agg(Flights=("Flights", "sum"), Revenue=("Revenue", "sum"),
    Flight_Costs=("Flight_Costs", "sum"), Gate_Costs=("Gate_Costs", "sum"),
    Profit_After_Gates=("Profit_After_Gates", "sum"))
by_origin["Margin_%"] = by_origin["Profit_After_Gates"] / by_origin["Revenue"] * 100
by_origin.sort_values("Profit_After_Gates")

,Flights,Revenue,Flight_Costs,Gate_Costs,Profit_After_Gates,Margin_%
Origin,,,,,,
YUL,1941,"18,137,870.59","36,594,965.33","1,171,331.59","-19,628,426.33",-108.22
YYZ,3259,"47,781,065.64","58,808,662.33","1,931,198.61","-12,958,795.30",-27.12
YHZ,1000,"8,613,540.02","17,078,611.45","724,736.28","-9,189,807.71",-106.69
YVR,2289,"44,618,005.29","50,975,597.13","1,856,017.59","-8,213,609.43",-18.41
YOW,1582,"4,413,429.48","10,067,121.84","444,394.04","-6,098,086.40",-138.17
YXU,530,"931,938.76","2,850,858.48","167,186.40","-2,086,106.12",-223.85
YQM,336,"584,758.53","1,804,597.78","159,456.00","-1,379,295.25",-235.87
YYT,168,"4,634,983.33","5,049,269.63","190,339.29","-604,625.59",-13.04
YZF,86,"1,462,152.06","1,817,498.46","153,688.18","-509,034.58",-34.81


**2d. Where did the expansion go?** (flights added vs profit change per route, FY25 -> FY26)

In [14]:
exp = (r25.set_index("Route_ID")[["Flights", "Revenue", "Profit_After_Gates"]]
       .join(r26.set_index("Route_ID")[["Flights", "Revenue", "Profit_After_Gates"]],
             lsuffix="_25", rsuffix="_26"))
exp["Flights_Added"]  = exp["Flights_26"] - exp["Flights_25"]
exp["Revenue_Change"] = exp["Revenue_26"] - exp["Revenue_25"]
exp["Profit_Change"]  = exp["Profit_After_Gates_26"] - exp["Profit_After_Gates_25"]
print(f"Routes that added flights: {(exp.Flights_Added>0).sum()} of {len(exp)}")
exp.sort_values("Flights_Added", ascending=False).head(12)[
    ["Flights_25", "Flights_26", "Flights_Added", "Revenue_Change", "Profit_Change"]]

Routes that added flights: 33 of 54


,Flights_25,Flights_26,Flights_Added,Revenue_Change,Profit_Change
Route_ID,,,,,
YUL-YOW,280,546,266,"597,393.98","-982,897.84"
YOW-YUL,284,544,260,"588,862.02","-882,090.39"
YYZ-YOW,807,1062,255,"79,875.25","-1,899,160.48"
YOW-YYZ,788,1038,250,"82,105.85","-1,940,260.74"
YYZ-YXU,314,534,220,"528,103.09","-816,262.27"
YXU-YYZ,316,530,214,"503,779.91","-717,297.53"
YQM-YHZ,144,336,192,"407,775.84","-640,023.79"
YHZ-YQM,144,332,188,"415,704.17","-704,535.27"
YYZ-YUL,481,634,153,"145,509.33","-2,239,457.01"


**Your answer (Q2):**

- Performance differences across the network: 
- Which parts carry the others: 

---
## Q3. How much do rising fuel and crew costs, refunds, and the NorthByte Rewards program contribute to the decline? Which factors matter most?

**3a. Ranking of drivers** (from the Q1 profit bridge; run Q1 first)

In [15]:
total_decline = delta_profit            # negative number
factors = pd.Series({
    "Flying 20% more flights (cost volume)":       bridge["More flights (volume of costs)"],
    "Fuel cost per flight":                         bridge["Higher fuel cost per flight"],
    "Crew cost per flight":                         bridge["Higher crew cost per flight"],
    "Maintenance + food cost per flight":           bridge["Higher maintenance cost per flight"] + bridge["Higher food cost per flight"],
    "Gate leases":                                  bridge["Gate leases"],
    "Refunds":                                      bridge["Refund revenue"],
    "Ticket revenue growth":                        bridge["Ticket revenue"],
    "Ancillary revenue growth (bags, seats, fees, food)":
        bridge[["Extra Baggage revenue", "Seat Upgrade revenue", "Change Fee revenue", "Food & Onboard revenue"]].sum(),
}, name="Profit_Impact")
assert abs(factors.sum() - total_decline) < 1

q3a = factors.sort_values().to_frame()
q3a["Share_of_Net_Decline_%"] = q3a["Profit_Impact"] / total_decline * 100
print(f"Net change in profit after gates: {total_decline:,.0f}")
q3a

Net change in profit after gates: -36,237,906


,Profit_Impact,Share_of_Net_Decline_%
Flying 20% more flights (cost volume),"-39,226,028.42",108.25
Crew cost per flight,"-2,911,262.41",8.03
Fuel cost per flight,"-2,490,486.91",6.87
Refunds,"-1,149,131.33",3.17
Maintenance + food cost per flight,"-763,134.52",2.11
Gate leases,"-216,525.91",0.60
"Ancillary revenue growth (bags, seats, fees, food)","292,539.12",-0.81
Ticket revenue growth,"10,226,124.46",-28.22


**3b. Refunds in more detail**

In [16]:
tickets = tx[tx.Transaction_Type == "Ticket"]
refunds = tx[tx.Transaction_Type == "Refund"]
ref_fy = pd.DataFrame({
    "Ticket_Revenue": tickets.groupby("Fiscal_Year")["Amount"].sum(),
    "Refunds":        refunds.groupby("Fiscal_Year")["Amount"].sum(),
    "Refund_Count":   refunds.groupby("Fiscal_Year").size(),
})
ref_fy["Refund_Rate_%_of_Tickets"] = -ref_fy["Refunds"] / ref_fy["Ticket_Revenue"] * 100
ref_fy["Avg_Refund"] = ref_fy["Refunds"] / ref_fy["Refund_Count"]
ref_fy

,Ticket_Revenue,Refunds,Refund_Count,Refund_Rate_%_of_Tickets,Avg_Refund
Fiscal_Year,,,,,
2025,"183,640,795.69","-6,684,921.18",6237,3.64,"-1,071.82"
2026,"193,866,920.15","-7,834,052.51",7283,4.04,"-1,075.66"


**3c. NorthByte Rewards** (the data has fares and tiers, not points liability, so we can test for fare leakage and mix shift only)

In [17]:
tiers_order = ["Non-Member", "Blue", "Silver", "Gold", "Platinum"]

tier = tickets.groupby(["Fiscal_Year", "Membership_Tier"]).agg(
    Tickets=("Amount", "size"), Avg_Fare=("Amount", "mean"), Ticket_Revenue=("Amount", "sum")).reset_index()
tier["Share_of_Tickets_%"] = tier["Tickets"] / tier.groupby("Fiscal_Year")["Tickets"].transform("sum") * 100

anc = tx[tx.Transaction_Type.isin(["Extra Baggage", "Seat Upgrade", "Change Fee", "Food & Onboard"])]
tier["Ancillary_per_Ticket"] = tier.apply(
    lambda r: anc[(anc.Fiscal_Year == r.Fiscal_Year) & (anc.Membership_Tier == r.Membership_Tier)]["Amount"].sum() / r.Tickets, axis=1)
tier["Refund_Rate_%"] = tier.apply(
    lambda r: -refunds[(refunds.Fiscal_Year == r.Fiscal_Year) & (refunds.Membership_Tier == r.Membership_Tier)]["Amount"].sum() / r.Ticket_Revenue * 100, axis=1)

tier["Membership_Tier"] = pd.Categorical(tier["Membership_Tier"], tiers_order, ordered=True)
display(tier.sort_values(["Fiscal_Year", "Membership_Tier"]))

print("Average fare by payment method (does points + cash change the fare?):")
display(tickets.groupby("Payment_Method")["Amount"].agg(["count", "mean"]))
member_share = tickets.assign(Member=np.where(tickets.Membership_Tier == "Non-Member", "Non-Member", "Member")) \
                      .groupby(["Fiscal_Year", "Member"])["Amount"].sum().unstack()
member_share["Member_Share_%"] = member_share["Member"] / member_share.sum(axis=1) * 100
member_share

,Fiscal_Year,Membership_Tier,Tickets,Avg_Fare,Ticket_Revenue,Share_of_Tickets_%,Ancillary_per_Ticket,Refund_Rate_%
2,2025,Non-Member,91017,"1,029.84","93,733,323.37",51.01,20.32,3.62
0,2025,Blue,43692,"1,026.76","44,861,237.23",24.49,20.51,3.72
4,2025,Silver,24207,"1,029.73","24,926,592.38",13.57,19.73,3.64
1,2025,Gold,14293,"1,032.76","14,761,286.02",8.01,23.12,3.49
3,2025,Platinum,5224,"1,025.72","5,358,356.69",2.93,23.13,3.82
7,2026,Non-Member,95467,"1,031.76","98,498,805.69",50.86,20.88,4.01
5,2026,Blue,46051,"1,032.89","47,565,748.25",24.53,20.81,4.17
9,2026,Silver,25767,"1,033.84","26,638,930.48",13.73,20.31,4.08
6,2026,Gold,14884,"1,034.51","15,397,589.48",7.93,24.15,3.90
8,2026,Platinum,5549,"1,039.08","5,765,846.25",2.96,23.76,3.65


Average fare by payment method (does points + cash change the fare?):


,count,mean
Payment_Method,,
Credit Card,236342,"1,030.40"
Debit,72087,"1,032.93"
Points + Cash,37106,"1,030.57"
Travel Voucher,17650,"1,030.08"
Unknown,2966,"1,044.94"


Member,Member,Non-Member,Member_Share_%
Fiscal_Year,,,
2025,"89,907,472.32","93,733,323.37",48.96
2026,"95,368,114.46","98,498,805.69",49.19


**Your answer (Q3):**

- Fuel & crew contribution: 
- Refunds contribution: 
- Rewards contribution: 
- Which factors matter most: 

---
## Q4. What role do ancillary revenue, refunds, and connecting passengers play in how NorthByte earns its money?

**4a. Revenue mix and ancillary**

In [18]:
anc_types = ["Extra Baggage", "Seat Upgrade", "Change Fee", "Food & Onboard"]
mix = rev_comp[["Ticket"] + anc_types + ["Refund"]].copy()
mix["Ancillary_Total"] = mix[anc_types].sum(axis=1)
mix["Net_Revenue"]     = mix[["Ticket", "Ancillary_Total", "Refund"]].sum(axis=1)
mix["Ancillary_%_of_Net_Revenue"] = mix["Ancillary_Total"] / mix["Net_Revenue"] * 100
mix["Refunds_%_of_Gross"]         = -mix["Refund"] / (mix["Ticket"] + mix["Ancillary_Total"]) * 100
growth = (mix.loc[2026] / mix.loc[2025] - 1) * 100
print("Growth FY25->FY26 (%):"); display(growth.round(1).to_frame("Growth_%").T)
mix.T

Growth FY25->FY26 (%):


Transaction_Type,Ticket,Extra Baggage,Seat Upgrade,Change Fee,Food & Onboard,Refund,Ancillary_Total,Net_Revenue,Ancillary_%_of_Net_Revenue,Refunds_%_of_Gross
Growth_%,5.60,6.90,7.50,8.20,13.30,17.20,8.00,5.20,2.60,11.00


Fiscal_Year,2025,2026
Transaction_Type,,
Ticket,"183,640,795.69","193,866,920.15"
Extra Baggage,"2,001,639.26","2,139,239.17"
Seat Upgrade,"678,561.23","729,536.24"
Change Fee,"558,125.00","603,975.00"
Food & Onboard,"435,546.30","493,660.50"
Refund,"-6,684,921.18","-7,834,052.51"
Ancillary_Total,"3,673,871.79","3,966,410.91"
Net_Revenue,"180,629,746.30","189,999,278.55"
Ancillary_%_of_Net_Revenue,2.03,2.09


In [19]:
n_tix = tickets.groupby("Fiscal_Year").size()
print("Ancillary $ per ticket:", (mix["Ancillary_Total"] / n_tix).round(2).to_dict())
print("Ticket revenue per flight:", (mix["Ticket"] / fy_summary["Flights"]).round(0).to_dict())
print("What if ancillary grew 10% / 25%? Profit impact FY26:",
      {f"+{p}%": round(mix.loc[2026, "Ancillary_Total"] * p / 100) for p in (10, 25)})

Ancillary $ per ticket: {2025: 20.59, 2026: 21.13}
Ticket revenue per flight: {2025: 14048.0, 2026: 12325.0}
What if ancillary grew 10% / 25%? Profit impact FY26: {'+10%': 396641, '+25%': 991603}


**4b. Refunds: where and when?** (refunds follow the >3-hour delay/cancellation promise; winter + Atlantic/North exposure)

In [20]:
REGION = {"YVR":"BC","YYJ":"BC","YLW":"BC","YXX":"BC","YXS":"BC",
          "YYC":"Prairies","YEG":"Prairies","YXE":"Prairies","YQR":"Prairies","YWG":"Prairies",
          "YYZ":"Central","YOW":"Central","YXU":"Central","YUL":"Central","YQB":"Central",
          "YHZ":"Atlantic","YQM":"Atlantic","YYT":"Atlantic","YXY":"North","YZF":"North"}
EXPOSED = {"Atlantic", "North"}

tx["Origin"], tx["Dest"] = tx["Route_ID"].str[:3], tx["Route_ID"].str[4:]
tx["Weather_Exposed"] = tx["Origin"].map(REGION).isin(EXPOSED) | tx["Dest"].map(REGION).isin(EXPOSED)
tx["Calendar_Month"]  = tx["Transaction_Date"].dt.month
tx["Winter"]          = tx["Calendar_Month"].isin([12, 1, 2, 3])

tk  = tx[tx.Transaction_Type == "Ticket"];  rf = tx[tx.Transaction_Type == "Refund"]

def refund_rate(group):
    t = tk.groupby(group)["Amount"].sum();  r = -rf.groupby(group)["Amount"].sum()
    return pd.DataFrame({"Tickets": t, "Refunds": r, "Refund_Rate_%": r / t * 100})

print("By exposure:");  display(refund_rate(["Fiscal_Year", "Weather_Exposed"]))
print("By winter vs other months:");  display(refund_rate(["Fiscal_Year", "Winter"]))
print("By month:");  display(refund_rate("Month").T)

By exposure:


Tickets      Refunds  Refund_Rate_%
Fiscal_Year Weather_Exposed                                           
2025        False           153,071,777.12 4,018,572.62           2.63
            True             30,569,018.57 2,666,348.56           8.72
2026        False           161,790,407.63 4,866,423.18           3.01
            True             32,076,512.52 2,967,629.33           9.25

By winter vs other months:


Tickets      Refunds  Refund_Rate_%
Fiscal_Year Winter                                           
2025        False  123,454,330.80 3,770,266.75           3.05
            True    60,186,464.89 2,914,654.43           4.84
2026        False  130,179,794.43 4,580,953.05           3.52
            True    63,687,125.72 3,253,099.46           5.11

By month:


Month,2024-09,2024-10,2024-11,2024-12,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08
Tickets,"13,803,416.69","14,165,678.88","14,084,051.31","16,177,895.28","14,837,048.98","14,047,638.80","15,123,881.83","13,811,016.00","15,133,425.89","16,857,905.43","18,220,247.16","17,378,589.44","14,979,623.27","15,062,363.20","15,558,417.36","17,207,857.32","15,743,444.57","14,826,319.90","15,909,503.93","14,435,197.74","15,772,602.08","17,493,825.79","18,883,247.19","17,994,517.80"
Refunds,"74,066.01","435,934.89","635,520.87","721,338.65","790,347.97","682,599.91","720,367.90","525,206.18","490,203.27","455,806.88","572,350.78","581,177.87","527,990.81","568,382.44","727,957.95","827,774.54","885,397.76","780,595.73","759,331.43","609,474.38","511,778.96","468,069.87","559,760.43","607,538.21"
Refund_Rate_%,0.54,3.08,4.51,4.46,5.33,4.86,4.76,3.80,3.24,2.70,3.14,3.34,3.52,3.77,4.68,4.81,5.62,5.26,4.77,4.22,3.24,2.68,2.96,3.38


In [21]:
rr_route = refund_rate(["Fiscal_Year", "Route_ID"]).reset_index()
rr_route = rr_route[rr_route.Fiscal_Year == 2026].sort_values("Refunds", ascending=False)
print("Routes with the most refund dollars, FY26:")
rr_route.head(10)

Routes with the most refund dollars, FY26:


,Fiscal_Year,Route_ID,Tickets,Refunds,Refund_Rate_%
98,2026,YYT-YYZ,"3,762,999.63","726,104.10",19.30
106,2026,YYZ-YYT,"3,620,173.30","711,116.44",19.64
80,2026,YVR-YYZ,"17,217,412.66","516,788.57",3.00
102,2026,YYZ-YVR,"18,674,374.29","504,293.13",2.70
97,2026,YYT-YHZ,"1,929,211.80","431,618.48",22.37
59,2026,YHZ-YYT,"1,540,685.15","332,472.73",21.58
71,2026,YUL-YVR,"9,522,611.96","314,761.80",3.31
105,2026,YYZ-YYC,"10,157,120.10","291,528.64",2.87
95,2026,YYC-YYZ,"8,897,256.29","291,256.93",3.27
75,2026,YVR-YUL,"8,651,536.40","258,912.92",2.99


**4c. Connecting passengers**

In [22]:
bk = tx.groupby(["Fiscal_Year", "Booking_ID"]).agg(
    Revenue=("Amount", "sum"), Booking_Type=("Booking_Type", "first")).reset_index()

conn = bk.groupby(["Fiscal_Year", "Booking_Type"]).agg(Bookings=("Booking_ID", "nunique"), Revenue=("Revenue", "sum"))
conn["Share_of_Bookings_%"] = conn["Bookings"] / conn.groupby("Fiscal_Year")["Bookings"].transform("sum") * 100
conn["Share_of_Revenue_%"]  = conn["Revenue"]  / conn.groupby("Fiscal_Year")["Revenue"].transform("sum") * 100
conn["Revenue_per_Booking"] = conn["Revenue"] / conn["Bookings"]
conn

Bookings        Revenue  Share_of_Bookings_%  Share_of_Revenue_%  Revenue_per_Booking
Fiscal_Year Booking_Type                                                                                       
2025        Connecting        6348  12,449,837.01                 3.69                6.89             1,961.22
            Direct          165615 168,179,909.29                96.31               93.11             1,015.49
2026        Connecting        6757  13,226,441.28                 3.65                6.96             1,957.44
            Direct          178147 176,772,837.27                96.35               93.04               992.29

In [23]:
# Which routes depend on connecting passengers? (matters before cutting any hub route)
dep = (tx[tx.Fiscal_Year == 2026].groupby(["Route_ID", "Booking_Type"])["Amount"].sum().unstack(fill_value=0))
dep["Connecting_Share_of_Route_Revenue_%"] = dep["Connecting"] / (dep["Connecting"] + dep["Direct"]) * 100
print("Where do connections happen? (airport shared between the two legs of connecting bookings)")
legs = tx[(tx.Booking_Type == "Connecting") & (tx.Transaction_Type == "Ticket")].groupby("Booking_ID")["Route_ID"].apply(list)
def hub(r):
    if len(r) < 2: return None
    a, b = r[0].split("-"), r[1].split("-")
    common = set(a) & set(b)
    return common.pop() if len(common) == 1 else None
print(legs.apply(hub).value_counts(dropna=False).head(6))
dep.sort_values("Connecting_Share_of_Route_Revenue_%", ascending=False).head(10)

Where do connections happen? (airport shared between the two legs of connecting bookings)
Route_ID
YYZ    7263
YYC    3852
YVR    1056
YUL     473
NaN     268
Name: count, dtype: int64


Booking_Type,Connecting,Direct,Connecting_Share_of_Route_Revenue_%
Route_ID,,,
YYC-YXX,"485,213.03","815,063.58",37.32
YXX-YYC,"468,967.23","812,480.21",36.60
YYC-YYZ,"2,822,735.17","5,891,334.68",32.39
YYZ-YYC,"2,912,772.06","7,076,950.14",29.16
YYZ-YUL,"742,517.50","3,447,132.53",17.72
YUL-YYZ,"703,532.41","3,724,086.33",15.89
YYZ-YHZ,"511,785.65","3,465,363.34",12.87
YHZ-YYZ,"555,375.29","4,212,977.94",11.65
YVR-YYZ,"1,129,460.30","15,770,000.79",6.68


**Your answer (Q4):**

- Ancillary role: 
- Refunds role: 
- Connecting passengers role: 

---
## Q5. How did gate lease costs change from the 2024-2025 to the 2025-2026 lease year? What would you expect NorthByte to pay in 2026-2027?

**5a. Change by airport**

In [24]:
q5 = gate_annual[["Lease_2425", "Lease_2526"]].copy()
q5["Change_$"] = q5["Lease_2526"] - q5["Lease_2425"]
q5["Change_%"] = q5["Change_$"] / q5["Lease_2425"] * 100
q5["Gates"]    = gate_unique[gate_unique.Year == "2025-2026"].groupby("City").size()
q5["Hub"]      = q5.index.isin(["Vancouver", "Calgary", "Toronto", "Montreal"])
q5.loc["TOTAL"] = q5.sum(numeric_only=True)
q5.loc["TOTAL", "Change_%"] = q5.loc["TOTAL", "Change_$"] / q5.loc["TOTAL", "Lease_2425"] * 100
q5.loc["TOTAL", "Hub"] = np.nan
q5.sort_values("Change_%", ascending=False)

Year,Lease_2425,Lease_2526,Change_$,Change_%,Gates,Hub
City,,,,,,
Toronto,"1,851,900.00","1,899,000.00","47,100.00",2.54,4.00,True
Montreal,"1,124,200.00","1,152,200.00","28,000.00",2.49,3.00,True
Vancouver,"1,782,300.00","1,826,100.00","43,800.00",2.46,4.00,True
Calgary,"1,065,800.00","1,090,600.00","24,800.00",2.33,3.00,True
TOTAL,"9,664,400.00","9,877,700.00","213,300.00",2.21,34.00,NaN
Winnipeg,"212,800.00","217,300.00","4,500.00",2.11,1.00,False
Halifax,"700,200.00","714,800.00","14,600.00",2.09,3.00,False
Edmonton,"490,700.00","500,800.00","10,100.00",2.06,2.00,False
Abbotsford,"166,800.00","170,200.00","3,400.00",2.04,1.00,False


**5b. Expected 2026-2027 cost** (three methods = a range; state which one you use)

In [25]:
tot_25, tot_26 = gate_annual["Lease_2425"].sum(), gate_annual["Lease_2526"].sum()
network_growth = tot_26 / tot_25 - 1

scenarios = pd.Series({
    "A. Each airport repeats its own % growth (base case)": gate_annual["Lease_2627_est"].sum(),
    "B. Network-average growth applied to every airport":   tot_26 * (1 + network_growth),
    "C. Flat (no increase)":                                tot_26,
    "D. Growth accelerates by 1 point":                     tot_26 * (1 + network_growth + 0.01),
}, name="Estimated_2026_27_Gate_Cost").to_frame()
scenarios["vs_2025_26_$"] = scenarios.iloc[:, 0] - tot_26
scenarios["vs_2025_26_%"] = scenarios["vs_2025_26_$"] / tot_26 * 100
print(f"2024-25: {tot_25:,.0f} | 2025-26: {tot_26:,.0f} | growth: {network_growth:.2%}")
scenarios

2024-25: 9,664,400 | 2025-26: 9,877,700 | growth: 2.21%


,Estimated_2026_27_Gate_Cost,vs_2025_26_$,vs_2025_26_%
A. Each airport repeats its own % growth (base case),"10,095,838.86","218,138.86",2.21
B. Network-average growth applied to every airport,"10,095,707.68","218,007.68",2.21
C. Flat (no increase),"9,877,700.00",0.00,0.00
D. Growth accelerates by 1 point,"10,194,484.68","316,784.68",3.21


**5c. Are we paying for gates we barely use?**

In [26]:
g26 = gate_unique[gate_unique.Year == "2025-2026"].copy()
routes_per_gate = gates[gates.Year == "2025-2026"].groupby(["City", "Gate_Number"])["Route_ID"].apply(lambda s: s.notna().sum())
g26 = g26.merge(routes_per_gate.rename("Routes_Assigned").reset_index(), on=["City", "Gate_Number"])
idle = g26[g26.Routes_Assigned == 0]
print("Gates with NO routes assigned (2025-26):")
display(idle[["City", "Gate_Number", "Annual_Gate_Cost"]])
idle_est = (idle.merge(gate_annual["Growth"].reset_index(), on="City")
                .assign(Est_2627=lambda d: d.Annual_Gate_Cost * (1 + d.Growth)))
print(f"Idle gate cost 2025-26: {idle.Annual_Gate_Cost.sum():,.0f}  |  expected 2026-27: {idle_est.Est_2627.sum():,.0f}")

Gates with NO routes assigned (2025-26):


,City,Gate_Number,Annual_Gate_Cost
8,Halifax,A3,225200
16,Ottawa,A2,208300


Idle gate cost 2025-26: 433,500  |  expected 2026-27: 441,963


In [27]:
util = pd.DataFrame({
    "Gates":  g26.groupby("City").size(),
    "Gate_Cost_2025_26": gate_annual["Lease_2526"],
    "Flights_FY26": fc[fc.Fiscal_Year == 2026].groupby("From").size(),
    "Flights_FY25": fc[fc.Fiscal_Year == 2025].groupby("From").size(),
})
util["Flights_per_Gate_FY26"]   = util["Flights_FY26"] / util["Gates"]
util["Gate_Cost_per_Flight_FY26"] = alloc[alloc.Fiscal_Year == 2026].set_index("City")["Gate_Per_Flight"]
util["Gate_Cost_per_Flight_FY25"] = alloc[alloc.Fiscal_Year == 2025].set_index("City")["Gate_Per_Flight"]
util.sort_values("Gate_Cost_per_Flight_FY26", ascending=False)

,Gates,Gate_Cost_2025_26,Flights_FY26,Flights_FY25,Flights_per_Gate_FY26,Gate_Cost_per_Flight_FY26,Gate_Cost_per_Flight_FY25
Whitehorse,1,154500,80,79,80.00,"1,950.71","1,945.99"
Yellowknife,1,152200,86,86,86.00,"1,787.07","1,761.24"
St. John's,1,188100,168,172,168.00,"1,132.97","1,087.21"
Abbotsford,1,170200,153,152,153.00,"1,127.54","1,112.28"
Regina,1,163500,153,153,153.00,"1,078.79","1,063.62"
Saskatoon,1,167900,165,166,165.00,"1,026.58","1,007.03"
Prince George,1,159000,159,157,159.00,"1,009.79","1,007.86"
Winnipeg,1,217300,243,246,243.00,906.85,877.24
Quebec City,1,174700,205,204,205.00,863.47,850.82
Vancouver,4,1826100,2289,2205,572.25,810.84,821.54


**Your answer (Q5):**

- How gate costs changed: 
- Expected 2026-27 cost and assumption: 
- Gate usage / idle gates: 

---
## Q6. What short-term and long-term actions would you recommend, and how would they affect customers, employees, and communities?

The cells below **size** the levers. Change the blue-sky parameters in the first cell and re-run. Every number is FY2025-26 run-rate and an *assumption you must state on the slide*.

In [28]:
# ===== EDIT THESE ASSUMPTIONS =====
CUT_SHARE       = 0.30    # share of flights removed on each targeted corridor
REV_RETAINED    = 0.60    # share of revenue on cut flights kept (passengers move to remaining flights)
N_CORRIDORS     = 8       # how many of the worst corridors to target
PROTECTED       = {"YXY", "YZF"}   # northern lifeline airports: never targeted (Connecting Communities value)
REFUND_REDUCTION= 0.10    # share of refunds avoided (schedule buffers, winter planning)
ANCILLARY_LIFT  = 0.10    # growth in ancillary revenue
FUEL_CREW_SAVING= 0.02    # efficiency saving on fuel + crew cost
# Share of each cost that actually disappears when a flight is cancelled
# (crew and maintenance are partly fixed: salaried crews, scheduled checks)
AVOIDABLE = {"Fuel": 1.00, "Crew": 0.80, "Maintenance": 0.90, "Food": 1.00}
# ==================================

**6a. Which corridors to cut back?** (corridor = both directions together, since aircraft fly round trips; gates are fixed so only flight costs are saved)

In [29]:
r26["Corridor"] = r26["Route_ID"].apply(lambda s: "-".join(sorted(s.split("-"))))
corr = r26.groupby("Corridor").agg(Flights=("Flights", "sum"), Revenue=("Revenue", "sum"),
        Flight_Costs=("Flight_Costs", "sum"), Gate_Costs=("Gate_Costs", "sum"),
        Profit_Before_Gates=("Profit_Before_Gates", "sum"), Profit_After_Gates=("Profit_After_Gates", "sum"))

fc26 = fc[fc.Fiscal_Year == 2026].copy()
fc26["Corridor"] = fc26["Route_ID"].apply(lambda r: "-".join(sorted(r.split("-"))))
fc26["Avoidable_Cost"] = sum(fc26[c] * w for c, w in AVOIDABLE.items())
corr["Avoidable_Cost"] = fc26.groupby("Corridor")["Avoidable_Cost"].sum()

# Connecting-passenger dependence of each corridor (cutting hub feed hurts other routes too)
dep_c = dep.reset_index()
dep_c["Corridor"] = dep_c["Route_ID"].apply(lambda s: "-".join(sorted(s.split("-"))))
conn_share = dep_c.groupby("Corridor")[["Connecting", "Direct"]].sum()
corr["Connecting_Share_%"] = conn_share["Connecting"] / (conn_share["Connecting"] + conn_share["Direct"]) * 100

eligible = corr[~corr.index.to_series().apply(lambda c: any(a in PROTECTED for a in c.split("-")))]
targets  = eligible.sort_values("Profit_Before_Gates").head(N_CORRIDORS).copy()

targets["Flights_Cut"]       = targets["Flights"] * CUT_SHARE
targets["Cost_Saved"]        = targets["Avoidable_Cost"] * CUT_SHARE
targets["Revenue_Lost"]      = targets["Revenue"] * CUT_SHARE * (1 - REV_RETAINED)
targets["Profit_Improvement"]= targets["Cost_Saved"] - targets["Revenue_Lost"]
print(f"Corridors in network: {len(corr)} | eligible (not protected): {len(eligible)}")
targets[["Flights", "Revenue", "Flight_Costs", "Profit_Before_Gates", "Connecting_Share_%",
         "Flights_Cut", "Cost_Saved", "Revenue_Lost", "Profit_Improvement"]]

Corridors in network: 27 | eligible (not protected): 25


,Flights,Revenue,Flight_Costs,Profit_Before_Gates,Connecting_Share_%,Flights_Cut,Cost_Saved,Revenue_Lost,Profit_Improvement
Corridor,,,,,,,,,
YUL-YVR,620,"17,812,769.73","37,518,516.59","-19,705,746.86",0.00,186.00,"10,282,932.13","2,137,532.37","8,145,399.76"
YHZ-YYZ,716,"8,745,502.22","17,894,077.94","-9,148,575.72",12.20,214.80,"4,869,120.36","1,049,460.27","3,819,660.09"
YUL-YYZ,1262,"8,617,268.77","17,041,963.17","-8,424,694.40",16.78,378.60,"4,619,378.52","1,034,072.25","3,585,306.26"
YOW-YYZ,2100,"6,761,807.37","13,955,860.90","-7,194,053.53",2.86,630.00,"3,778,320.46","811,416.88","2,966,903.58"
YHZ-YUL,452,"3,866,353.13","8,089,678.95","-4,223,325.82",4.14,135.60,"2,196,262.67","463,962.38","1,732,300.29"
YOW-YUL,1090,"1,915,155.60","5,862,183.54","-3,947,027.94",0.00,327.00,"1,583,268.21","229,818.67","1,353,449.54"
YXU-YYZ,1064,"1,886,190.47","5,757,742.43","-3,871,551.96",0.00,319.20,"1,554,535.84","226,342.86","1,328,192.98"
YHZ-YQM,668,"1,179,140.55","3,611,915.37","-2,432,774.82",0.00,200.40,"976,391.55","141,496.87","834,894.68"


In [30]:
# Sensitivity: how much does the answer depend on the two big assumptions?
grid = pd.DataFrame(
    {f"retain {int(rr*100)}%": [(targets["Avoidable_Cost"].sum() * cs) - (targets["Revenue"].sum() * cs * (1 - rr))
                                for cs in (0.15, 0.30, 0.50)] for rr in (0.4, 0.6, 0.8)},
    index=[f"cut {int(cs*100)}% of flights" for cs in (0.15, 0.30, 0.50)])
print("Profit improvement from corridor cuts (annual $):")
grid

Profit improvement from corridor cuts (annual $):


,retain 40%,retain 60%,retain 80%
cut 15% of flights,"10,359,527.96","11,883,053.59","13,406,579.23"
cut 30% of flights,"20,719,055.92","23,766,107.19","26,813,158.46"
cut 50% of flights,"34,531,759.87","39,610,178.65","44,688,597.43"


**6b. All levers together**

In [31]:
fy26_rev = rev_comp.loc[2026]
fy26_fuel_crew = fc[fc.Fiscal_Year == 2026][["Fuel", "Crew"]].sum().sum()

levers = pd.DataFrame([
    ["Reduce frequency on weakest corridors",          targets["Profit_Improvement"].sum(),
        f"Cut {CUT_SHARE:.0%} of flights on {N_CORRIDORS} corridors, keep {REV_RETAINED:.0%} of revenue; excludes northern routes"],
    ["Release unused gates (Halifax A3, Ottawa A2)",   idle_est["Est_2627"].sum(),
        "No routes assigned; est. 2026-27 lease"],
    ["Cut avoidable refunds",                          -fy26_rev["Refund"] * REFUND_REDUCTION,
        f"{REFUND_REDUCTION:.0%} fewer refunds via winter scheduling/buffers"],
    ["Grow ancillary revenue",                         fy26_rev[anc_types].sum() * ANCILLARY_LIFT,
        f"+{ANCILLARY_LIFT:.0%} on bags, seats, change fees, food"],
    ["Fuel and crew efficiency",                       fy26_fuel_crew * FUEL_CREW_SAVING,
        f"{FUEL_CREW_SAVING:.0%} of fuel + crew (routing, rostering)"],
], columns=["Lever", "Annual_Profit_Impact", "Assumption"])

baseline = fy_summary.loc[2026, "Profit_After_Gates"]
levers.loc[len(levers)] = ["TOTAL", levers["Annual_Profit_Impact"].sum(), ""]
print(f"FY2025-26 profit after gates: {baseline:,.0f}")
print(f"Pro-forma after levers:       {baseline + levers.iloc[-1]['Annual_Profit_Impact']:,.0f}")
print(f"Gap still to close:           {-(baseline + levers.iloc[-1]['Annual_Profit_Impact']):,.0f}")
levers

FY2025-26 profit after gates: -58,400,324
Pro-forma after levers:       -29,853,218
Gap still to close:           29,853,218


,Lever,Annual_Profit_Impact,Assumption
0,Reduce frequency on weakest corridors,"23,766,107.19","Cut 30% of flights on 8 corridors, keep 60% of..."
1,"Release unused gates (Halifax A3, Ottawa A2)","441,962.76",No routes assigned; est. 2026-27 lease
2,Cut avoidable refunds,"783,405.25",10% fewer refunds via winter scheduling/buffers
3,Grow ancillary revenue,"396,641.09","+10% on bags, seats, change fees, food"
4,Fuel and crew efficiency,"3,158,989.61","2% of fuel + crew (routing, rostering)"
5,TOTAL,"28,547,105.91",


**6c. What happens if we remove the weakest routes? (route-removal simulator)**

Rules baked in: (1) routes are removed as **corridors** (both directions) because aircraft fly round trips; (2) **gate leases do not shrink**, so the saving is *avoidable flight cost minus lost revenue*, never "profit after gates"; (3) you choose how much revenue is **retained** (passengers rebooked onto remaining flights); (4) a **connectivity check** flags any airport left without hub service.

In [32]:
def build_graph(corridor_list):
    g = {}
    for c in corridor_list:
        a, b = c.split("-")
        g.setdefault(a, set()).add(b); g.setdefault(b, set()).add(a)
    return g

def direct_hub_links(corridor_list):
    g = build_graph(corridor_list)
    return {a for a, nb in g.items() if a not in HUBS and (nb & set(HUBS))}

def stranded(removed, all_corridors):
    """Compared with today's network: airports that lose ALL service, or lose their DIRECT hub link."""
    remaining = [c for c in all_corridors if c not in set(removed)]
    airports = {a for c in all_corridors for a in c.split("-")}
    served_after = set(build_graph(remaining))
    no_service = sorted(a for a in airports if a not in served_after)
    lost_hub = sorted((direct_hub_links(all_corridors) - direct_hub_links(remaining)) - set(no_service))
    return no_service, lost_hub

def simulate_removal(corridors, cut_share=1.0, rev_retained=0.0):
    """Impact on FY26 run-rate profit of cutting `cut_share` of flights on the given corridors."""
    d = corr.loc[corridors]
    cost_saved   = (d["Avoidable_Cost"] * cut_share).sum()
    revenue_lost = (d["Revenue"] * cut_share * (1 - rev_retained)).sum()
    out = {"Corridors": len(d), "Flights_Removed": d["Flights"].sum() * cut_share,
           "Avoidable_Cost_Saved": cost_saved, "Revenue_Lost": revenue_lost,
           "Profit_Change": cost_saved - revenue_lost}
    out["New_Profit_After_Gates"] = fy_summary.loc[2026, "Profit_After_Gates"] + out["Profit_Change"]
    if cut_share >= 1.0:
        out["Lose_All_Service"], out["Lose_Direct_Hub_Link"] = stranded(corridors, list(corr.index))
    return out

# Example: remove YUL-YVR completely, keeping 50% of its revenue on other flights
simulate_removal(["YUL-YVR"], cut_share=1.0, rev_retained=0.5)

{'Corridors': 1,
 'Flights_Removed': np.float64(620.0),
 'Avoidable_Cost_Saved': np.float64(34276440.43),
 'Revenue_Lost': np.float64(8906384.865),
 'Profit_Change': np.float64(25370055.564999998),
 'New_Profit_After_Gates': np.float64(-33030268.04288628),
 'Lose_All_Service': [],
 'Lose_Direct_Hub_Link': []}

**Remove the worst K corridors entirely** (ranked by contribution = revenue - flight costs, because gates are fixed)

In [33]:
ranked = eligible.sort_values("Profit_Before_Gates").index.tolist()   # eligible = excludes PROTECTED northern airports
rows = []
for k in range(1, 11):
    chosen = ranked[:k]
    row = {"K": k, "Last_Corridor_Added": chosen[-1]}
    for rr in (0.0, 0.4, 0.8):
        row[f"Profit_Change (retain {int(rr*100)}%)"] = simulate_removal(chosen, 1.0, rr)["Profit_Change"]
    sim_ = simulate_removal(chosen, 1.0, 0.4)
    row["Lose_All_Service"] = ", ".join(sim_["Lose_All_Service"]) or "-"
    row["Lose_Direct_Hub_Link"] = ", ".join(sim_["Lose_Direct_Hub_Link"]) or "-"
    rows.append(row)
removal_curve = pd.DataFrame(rows).set_index("K")
removal_curve

,Last_Corridor_Added,Profit_Change (retain 0%),Profit_Change (retain 40%),Profit_Change (retain 80%),Lose_All_Service,Lose_Direct_Hub_Link
K,,,,,,
1,YUL-YVR,"16,463,670.70","23,588,778.59","30,713,886.48",-,-
2,YHZ-YYZ,"23,948,569.67","34,571,878.45","45,195,187.23",-,-
3,YUL-YYZ,"30,729,229.29","44,799,445.58","58,869,661.86",-,-
4,YOW-YYZ,"36,561,823.47","53,336,762.70","70,111,701.94",-,-
5,YHZ-YUL,"40,016,345.89","58,337,826.38","76,659,306.87",-,YHZ
6,YOW-YUL,"43,378,750.99","62,466,293.72","81,553,836.45",YOW,YHZ
7,YXU-YYZ,"46,674,346.64","66,516,365.56","86,358,384.48","YOW, YXU",YHZ
8,YHZ-YQM,"48,749,844.59","69,063,519.73","89,377,194.87","YOW, YQM, YXU",YHZ
9,YYT-YYZ,"48,576,361.58","71,312,822.94","94,049,284.29","YOW, YQM, YXU","YHZ, YYT"


**Upper bound: remove EVERY corridor with negative contribution** (not a recommendation; it shows the size of the prize and what it would do to the network)

In [34]:
neg = eligible[eligible["Profit_Before_Gates"] < 0].index.tolist()   # protected northern corridors excluded
for rr in (0.0, 0.4, 0.8):
    r = simulate_removal(neg, 1.0, rr)
    print(f"retain {rr:.0%}: profit change {r['Profit_Change']:>14,.0f} | new profit after gates {r['New_Profit_After_Gates']:>14,.0f}")
print("\nCorridors removed:", len(neg), "of", len(corr), "| northern/protected corridors kept")
ns, nh = stranded(neg, list(corr.index))
print("Airports left with NO service:", ns or "none")
print("Airports that lose their direct hub link:", nh or "none")

retain 0%: profit change     48,555,099 | new profit after gates     -9,845,225
retain 40%: profit change     74,379,017 | new profit after gates     15,978,693
retain 80%: profit change    100,202,935 | new profit after gates     41,802,611

Corridors removed: 12 of 27 | northern/protected corridors kept
Airports left with NO service: ['YHZ', 'YOW', 'YQM', 'YXS', 'YXU', 'YXX', 'YYT']
Airports that lose their direct hub link: none


**Better than removal: partial frequency cuts** (cut X% of flights, keep the corridor). Sensitivity to the two assumptions that matter most:

In [35]:
tgt = targets.index.tolist()
sens = pd.DataFrame(
    {f"retain {int(rr*100)}%": [simulate_removal(tgt, cs, rr)["Profit_Change"] for cs in (0.15, 0.30, 0.50, 1.00)]
     for rr in (0.0, 0.4, 0.6, 0.8)},
    index=[f"cut {int(cs*100)}% of flights" for cs in (0.15, 0.30, 0.50, 1.00)])
print(f"Targeted corridors ({len(tgt)}):", ", ".join(tgt))
sens

Targeted corridors (8): YUL-YVR, YHZ-YYZ, YUL-YYZ, YOW-YYZ, YHZ-YUL, YOW-YUL, YXU-YYZ, YHZ-YQM


,retain 0%,retain 40%,retain 60%,retain 80%
cut 15% of flights,"7,312,476.69","10,359,527.96","11,883,053.59","13,406,579.23"
cut 30% of flights,"14,624,953.38","20,719,055.92","23,766,107.19","26,813,158.46"
cut 50% of flights,"24,374,922.30","34,531,759.87","39,610,178.65","44,688,597.43"
cut 100% of flights,"48,749,844.59","69,063,519.73","79,220,357.30","89,377,194.87"


**6d. Impact on customers, employees, communities** (fill in using the outputs above)

| Lever | Customers | Employees | Communities |
|---|---|---|---|
| Corridor frequency cuts | | | |
| Release idle gates | | | |
| Refund reduction | | | |
| Ancillary growth | | | |
| Fuel / crew efficiency | | | |

**Your answer (Q6):**

- Short-term actions (0-12 months): 
- Long-term actions (1-3 years): 
- What we protect (safety, northern/regional service) and why: 